# Case 5 — KL free-bits sweet spot

**Reproduces:** Fig 4.13, Table 4.8 (section c)

Transformer-VAE, contextual anomalies. Transformer-VAE's expressive decoder can partly reconstruct without leaning on the latent code, so free_bits=0 is expected to under-use z. A moderate floor (0.5, the baseline) forces every latent dimension to carry some information; too large a floor (2.0) is expected to add noise back in. Expect the middle value to win — not the extremes.

Runtime menu -> Change runtime type -> GPU, then run all cells.

This notebook runs on the real ERA5 slice shipped with the repo under `data/era5/` (16 years, centered on the same warmup/test split used at full scale, split exactly 50:50 warmup/test — see `DATA_LICENSE.md`) — no download needed. Numbers will still differ from the thesis's full-scale figures (much shorter warmup/test period, noisier), but the *qualitative* effect described above should still show up.

**Setup:** this is a private repo, so before running you need a GitHub token as a Colab secret — key icon in the left sidebar -> new secret named `GITHUB_TOKEN`, value = a token from [github.com/settings/tokens](https://github.com/settings/tokens) (read-only `repo` access is enough), then toggle "Notebook access" on.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

In [ ]:
import os

REPO_URL = "https://github.com/hadasecohen/streaming-vae-anomaly-detection.git"

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Private repo: add a GitHub token as a Colab secret first --
    # key icon in the left sidebar -> Secrets -> new secret named GITHUB_TOKEN,
    # value = a token from github.com/settings/tokens (read-only "repo" access is enough)
    # -> toggle "Notebook access" on.
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    clone_url = REPO_URL.replace("https://", f"https://{token}@")
    if not os.path.exists("repo"):
        !git clone $clone_url repo
    %cd repo
elif not os.path.exists("run_regression.py"):
    # Already inside a local checkout (e.g. running from notebooks/cases/) --
    # move to the repo root instead of cloning a redundant nested copy.
    %cd ../..

!pip install -q -r requirements.txt


## Run the suite

`notebooks/cases/case05_free_bits_suite.yaml` layers `modules/era5_common.yaml` with the architecture / anomaly-type / stream-mode / feature-engineering fragments for each of the runs below (see the suite file for the exact overrides). Runs execute sequentially. Window-mode runs are batched and fast (well under a minute each on GPU); point-mode runs stream one gradient step per row with no batching, so they run on CPU instead (faster than GPU for this access pattern) and take a few minutes each — `modules/stream/point.yaml` caps them to a 5,000-row subset for this reason.

In [ ]:
!python run_regression.py notebooks/cases/case05_free_bits_suite.yaml \
    --session runs/regression/case05_free_bits

## Compare the runs

`cross_compare.py` walks the session directory, extracts F1/AUC/precision/recall from each run's `trial_predictions.csv`, and writes a performance table plus comparison plots (score histograms, F1-vs-variant line plot, confusion grid, seed-stability heatmap) under `<session>/cross_compare/`.

In [ ]:
!python cross_compare.py runs/regression/case05_free_bits

In [ ]:
import pandas as pd
perf = pd.read_csv("runs/regression/case05_free_bits/cross_compare/performance_table.csv")
perf[["run_name", "arch", "anomaly", "variant", "f1", "auc", "precision", "recall", "tp", "fp", "fn"]]

In [ ]:
# Display the key comparison plot(s) inline
import glob
from IPython.display import Image, display

print("F1 vs free_bits floor:")
for p in sorted(glob.glob("runs/regression/case05_free_bits/cross_compare/contextual/section_lines_free_bits.png")):
    display(Image(filename=p))